# imports

In [1]:
from pathlib import Path
from typing import Optional, Any, Union

import numpy as np
import polars as pl

# plotting
import matplotlib.pyplot as plt
import plotly.express as px
from dash import Dash, dcc, html, Input, Output, State, callback_context
import plotly.graph_objects as go

# scipy
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

# muutils
import muutils.tensor_info
from muutils.dbg import dbg, dbg_tensor

# attention-motifs
from attention_motifs.features.analysis import nan_stats, filter_data, normalize_data
from attention_motifs.features.plotting import (
	plot_correlation_matrix,
	plot_embedding,
	apply_pca,
)

muutils.tensor_info.DEFAULT_SETTINGS["colored"] = True

from pathlib import Path
from typing import Optional, Any, Union

import numpy as np
from jaxtyping import Float

import polars as pl

# muutils
from muutils.jsonlines import jsonl_write, jsonl_load

# attention-motifs
from attention_motifs.bins import Bins
from attention_motifs.features.features import scalar_feature_table
from attention_motifs.features.hist_beta_fit import hist_beta_fit
from attention_motifs.util import prefix_dict
from attention_motifs.features.transition_tensor import tt_features
from attention_motifs.features.vec_features import vec_features
from attention_motifs.math.cos_sim import cosine_similarity_matrix
from attention_motifs.math.math import skew_lt
from attention_motifs.features.analysis import nan_stats, filter_data, normalize_data

# set up feature generation

In [2]:
def gram_features(A: Float[np.ndarray, "n_ctx n_ctx"]) -> dict[str, float]:
	# dbg_tensor(A)
	return prefix_dict(
		hist_beta_fit(
			A.flatten(),
			bins=Bins(n_bins=32, start=0.0, stop=1.0),
		),
		prefix="beta_hist",
	)
	# TODO: mass as a function of distance from diagonal

In [3]:
def compute_scalar_features(
	A: Float[np.ndarray, "n_ctx n_ctx"],
) -> dict[str, float]:
	# dbg_tensor(A)
	A_log: Float[np.ndarray, "n_ctx n_ctx"] = np.nan_to_num(np.log(A + 1e-9), nan=-10)
	# dbg_tensor(A_log)

	A_skew: Float[np.ndarray, "n_ctx n_ctx"] = skew_lt(A)
	# dbg_tensor(A_skew)
	A_log_skew: Float[np.ndarray, "n_ctx n_ctx"] = skew_lt(A_log)

	return dict(
		# diagonal: standard features, fit diff to beta dist
		**prefix_dict(vec_features(A.diagonal()), prefix="diag"),
		# off-diagonal: standard features, fit diff to beta dist
		**prefix_dict(vec_features(A[:, 0]), prefix="first_tok"),
		# transition tensor: standard features, standard features on diff, linear envelope on transition time
		# 	TODO: standard features on decay rate
		**prefix_dict(
			tt_features(A),
			prefix="markov_transition",
		),
		# # {log, raw} gram matrix of {rows, cols, rows of skewed}: beta fit hist
		# # 	TODO: fit fft in `gram_features`, but this is expensive
		**prefix_dict(
			gram_features(A @ A.T),
			prefix=["gram", "row"],
		),
		**prefix_dict(
			gram_features(A.T @ A),
			prefix=["gram", "col"],
		),
		**prefix_dict(
			gram_features(A_skew.T @ A_skew),
			prefix=["gram", "skew"],
		),
		**prefix_dict(
			gram_features(cosine_similarity_matrix(A_log)),
			prefix=["log", "gram", "row"],
		),
		**prefix_dict(
			gram_features(cosine_similarity_matrix(A_log, col=True)),
			prefix=["log", "gram", "col"],
		),
		**prefix_dict(
			gram_features(cosine_similarity_matrix(A_log_skew)),
			prefix=["log", "gram", "skew"],
		),
	)

# run feature gen

In [4]:
DATA_RAW: pl.DataFrame = scalar_feature_table(
	features_func=compute_scalar_features,
	models=["pythia-14m", "tiny-stories-1M"],
)

models: ['pythia-14m', 'tiny-stories-1M']
model: 'pythia-14m'
✔️  (0.00s) setting up paths                                                   
✔️  (0.01s) loading prompts                                                    
128 prompts loaded


100%|██████████| 128/128 [01:02<00:00,  2.06it/s]


model: 'tiny-stories-1M'
✔️  (0.00s) setting up paths                                                   
✔️  (0.00s) loading prompts                                                    
128 prompts loaded


100%|██████████| 128/128 [05:56<00:00,  2.79s/it]


# save raw data

In [43]:
DATA_RAW.write_ndjson(Path("../data/features/raw.jsonl"))

# basic data inspection

In [44]:
DATA_RAW

activation.model,activation.layer,activation.cache_key,activation.head,activation.cls,activation.prompt,activation.n_ctx,feat.diag.mean,feat.diag.median,feat.diag.variance,feat.diag.std,feat.diag.skewness,feat.diag.kurtosis,feat.diag.entropy,feat.diag.L1_norm,feat.diag.L2_norm,feat.diag.rms,feat.diag.energy,feat.diag.zero_crossing_rate,feat.diag.autocorr_lag1,feat.diag.psd_total_power,feat.diag.linreg.slope,feat.diag.linreg.intercept,feat.diag.linreg.r2,feat.first_tok.mean,feat.first_tok.median,feat.first_tok.variance,feat.first_tok.std,feat.first_tok.skewness,feat.first_tok.kurtosis,feat.first_tok.entropy,feat.first_tok.L1_norm,feat.first_tok.L2_norm,feat.first_tok.rms,feat.first_tok.energy,feat.first_tok.zero_crossing_rate,feat.first_tok.autocorr_lag1,…,feat.log.gram.row.beta_hist.hist.raw.linreg.slope,feat.log.gram.row.beta_hist.hist.raw.linreg.intercept,feat.log.gram.row.beta_hist.hist.raw.linreg.r2,feat.log.gram.col.beta_hist.hist.raw.mean,feat.log.gram.col.beta_hist.hist.raw.median,feat.log.gram.col.beta_hist.hist.raw.variance,feat.log.gram.col.beta_hist.hist.raw.std,feat.log.gram.col.beta_hist.hist.raw.skewness,feat.log.gram.col.beta_hist.hist.raw.kurtosis,feat.log.gram.col.beta_hist.hist.raw.entropy,feat.log.gram.col.beta_hist.hist.raw.L1_norm,feat.log.gram.col.beta_hist.hist.raw.L2_norm,feat.log.gram.col.beta_hist.hist.raw.rms,feat.log.gram.col.beta_hist.hist.raw.energy,feat.log.gram.col.beta_hist.hist.raw.zero_crossing_rate,feat.log.gram.col.beta_hist.hist.raw.autocorr_lag1,feat.log.gram.col.beta_hist.hist.raw.psd_total_power,feat.log.gram.col.beta_hist.hist.raw.linreg.slope,feat.log.gram.col.beta_hist.hist.raw.linreg.intercept,feat.log.gram.col.beta_hist.hist.raw.linreg.r2,feat.log.gram.skew.beta_hist.hist.raw.mean,feat.log.gram.skew.beta_hist.hist.raw.median,feat.log.gram.skew.beta_hist.hist.raw.variance,feat.log.gram.skew.beta_hist.hist.raw.std,feat.log.gram.skew.beta_hist.hist.raw.skewness,feat.log.gram.skew.beta_hist.hist.raw.kurtosis,feat.log.gram.skew.beta_hist.hist.raw.entropy,feat.log.gram.skew.beta_hist.hist.raw.L1_norm,feat.log.gram.skew.beta_hist.hist.raw.L2_norm,feat.log.gram.skew.beta_hist.hist.raw.rms,feat.log.gram.skew.beta_hist.hist.raw.energy,feat.log.gram.skew.beta_hist.hist.raw.zero_crossing_rate,feat.log.gram.skew.beta_hist.hist.raw.autocorr_lag1,feat.log.gram.skew.beta_hist.hist.raw.psd_total_power,feat.log.gram.skew.beta_hist.hist.raw.linreg.slope,feat.log.gram.skew.beta_hist.hist.raw.linreg.intercept,feat.log.gram.skew.beta_hist.hist.raw.linreg.r2
str,i64,str,i64,str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""pythia-14m""",0,"""blocks.0.attn.hook_pattern""",0,"""pythia-14m:L0:H0""","""MjWkadUxH4Z5nfLEFtUc1w""",114,0.169934,0.127726,0.027237,0.165037,1.939376,5.722651,2.15858,0.169934,0.022139,0.236381,6.369844,0.0,0.158451,1.596638,-0.001612,0.261002,0.104208,0.076226,0.007402,0.028752,0.169565,3.412402,12.67031,1.140814,0.076226,0.017348,0.185231,3.911398,0.0,0.782999,…,0.029276,-0.290582,0.133926,0.164973,0.0,0.529036,0.727349,4.743038,21.737763,0.39979,0.164973,0.12987,0.734656,17.271027,0.0,0.961123,0.878773,0.02949,-0.292124,0.144661,0.931865,0.22674,9.354491,3.058511,5.208675,25.775427,0.200622,0.931865,0.557072,3.151275,317.777149,0.0,-0.025481,16.549212,-0.055552,1.792928,0.029031
"""pythia-14m""",0,"""blocks.0.attn.hook_pattern""",1,"""pythia-14m:L0:H1""","""MjWkadUxH4Z5nfLEFtUc1w""",114,0.251202,0.123246,0.08036,0.283479,0.99125,-0.29873,2.593486,0.251202,0.035387,0.377833,16.274387,0.0,-0.096441,7.588164,-0.00052,0.280562,0.003671,0.039348,0.000129,0.023648,0.153779,5.195867,27.259118,0.560997,0.039348,0.014805,0.158078,2.848719,0.0,0.087,…,0.028574,-0.285403,0.096994,0.159597,0.0,0.666917,0.81665,5.298928,26.369746,0.39979,0.159597,0.144865,0.81948

In [45]:
set(DATA_RAW.dtypes)

{Float64, Int64, String}

In [46]:
DATA_RAW.head()

activation.model,activation.layer,activation.cache_key,activation.head,activation.cls,activation.prompt,activation.n_ctx,feat.diag.mean,feat.diag.median,feat.diag.variance,feat.diag.std,feat.diag.skewness,feat.diag.kurtosis,feat.diag.entropy,feat.diag.L1_norm,feat.diag.L2_norm,feat.diag.rms,feat.diag.energy,feat.diag.zero_crossing_rate,feat.diag.autocorr_lag1,feat.diag.psd_total_power,feat.diag.linreg.slope,feat.diag.linreg.intercept,feat.diag.linreg.r2,feat.first_tok.mean,feat.first_tok.median,feat.first_tok.variance,feat.first_tok.std,feat.first_tok.skewness,feat.first_tok.kurtosis,feat.first_tok.entropy,feat.first_tok.L1_norm,feat.first_tok.L2_norm,feat.first_tok.rms,feat.first_tok.energy,feat.first_tok.zero_crossing_rate,feat.first_tok.autocorr_lag1,…,feat.log.gram.row.beta_hist.hist.raw.linreg.slope,feat.log.gram.row.beta_hist.hist.raw.linreg.intercept,feat.log.gram.row.beta_hist.hist.raw.linreg.r2,feat.log.gram.col.beta_hist.hist.raw.mean,feat.log.gram.col.beta_hist.hist.raw.median,feat.log.gram.col.beta_hist.hist.raw.variance,feat.log.gram.col.beta_hist.hist.raw.std,feat.log.gram.col.beta_hist.hist.raw.skewness,feat.log.gram.col.beta_hist.hist.raw.kurtosis,feat.log.gram.col.beta_hist.hist.raw.entropy,feat.log.gram.col.beta_hist.hist.raw.L1_norm,feat.log.gram.col.beta_hist.hist.raw.L2_norm,feat.log.gram.col.beta_hist.hist.raw.rms,feat.log.gram.col.beta_hist.hist.raw.energy,feat.log.gram.col.beta_hist.hist.raw.zero_crossing_rate,feat.log.gram.col.beta_hist.hist.raw.autocorr_lag1,feat.log.gram.col.beta_hist.hist.raw.psd_total_power,feat.log.gram.col.beta_hist.hist.raw.linreg.slope,feat.log.gram.col.beta_hist.hist.raw.linreg.intercept,feat.log.gram.col.beta_hist.hist.raw.linreg.r2,feat.log.gram.skew.beta_hist.hist.raw.mean,feat.log.gram.skew.beta_hist.hist.raw.median,feat.log.gram.skew.beta_hist.hist.raw.variance,feat.log.gram.skew.beta_hist.hist.raw.std,feat.log.gram.skew.beta_hist.hist.raw.skewness,feat.log.gram.skew.beta_hist.hist.raw.kurtosis,feat.log.gram.skew.beta_hist.hist.raw.entropy,feat.log.gram.skew.beta_hist.hist.raw.L1_norm,feat.log.gram.skew.beta_hist.hist.raw.L2_norm,feat.log.gram.skew.beta_hist.hist.raw.rms,feat.log.gram.skew.beta_hist.hist.raw.energy,feat.log.gram.skew.beta_hist.hist.raw.zero_crossing_rate,feat.log.gram.skew.beta_hist.hist.raw.autocorr_lag1,feat.log.gram.skew.beta_hist.hist.raw.psd_total_power,feat.log.gram.skew.beta_hist.hist.raw.linreg.slope,feat.log.gram.skew.beta_hist.hist.raw.linreg.intercept,feat.log.gram.skew.beta_hist.hist.raw.linreg.r2
str,i64,str,i64,str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""pythia-14m""",0,"""blocks.0.attn.hook_pattern""",0,"""pythia-14m:L0:H0""","""MjWkadUxH4Z5nfLEFtUc1w""",114,0.169934,0.127726,0.027237,0.165037,1.939376,5.722651,2.15858,0.169934,0.022139,0.236381,6.369844,0.0,0.158451,1.596638,-0.001612,0.261002,0.104208,0.076226,0.007402,0.028752,0.169565,3.412402,12.67031,1.140814,0.076226,0.017348,0.185231,3.911398,0.0,0.782999,…,0.029276,-0.290582,0.133926,0.164973,0.0,0.529036,0.727349,4.743038,21.737763,0.39979,0.164973,0.12987,0.734656,17.271027,0.0,0.961123,0.878773,0.02949,-0.292124,0.144661,0.931865,0.22674,9.354491,3.058511,5.208675,25.775427,0.200622,0.931865,0.557072,3.151275,317.777149,0.0,-0.025481,16.549212,-0.055552,1.792928,0.029031
"""pythia-14m""",0,"""blocks.0.attn.hook_pattern""",1,"""pythia-14m:L0:H1""","""MjWkadUxH4Z5nfLEFtUc1w""",114,0.251202,0.123246,0.08036,0.283479,0.99125,-0.29873,2.593486,0.251202,0.035387,0.377833,16.274387,0.0,-0.096441,7.588164,-0.00052,0.280562,0.003671,0.039348,0.000129,0.023648,0.153779,5.195867,27.259118,0.560997,0.039348,0.014805,0.158078,2.848719,0.0,0.087,…,0.028574,-0.285403,0.096994,0.159597,0.0,0.666917,0.81665,5.298928,26.369746,0.39979,0.159597,0.144865,0.81948

In [47]:
DATA_RAW.describe()

statistic,activation.model,activation.layer,activation.cache_key,activation.head,activation.cls,activation.prompt,activation.n_ctx,feat.diag.mean,feat.diag.median,feat.diag.variance,feat.diag.std,feat.diag.skewness,feat.diag.kurtosis,feat.diag.entropy,feat.diag.L1_norm,feat.diag.L2_norm,feat.diag.rms,feat.diag.energy,feat.diag.zero_crossing_rate,feat.diag.autocorr_lag1,feat.diag.psd_total_power,feat.diag.linreg.slope,feat.diag.linreg.intercept,feat.diag.linreg.r2,feat.first_tok.mean,feat.first_tok.median,feat.first_tok.variance,feat.first_tok.std,feat.first_tok.skewness,feat.first_tok.kurtosis,feat.first_tok.entropy,feat.first_tok.L1_norm,feat.first_tok.L2_norm,feat.first_tok.rms,feat.first_tok.energy,feat.first_tok.zero_crossing_rate,…,feat.log.gram.row.beta_hist.hist.raw.linreg.slope,feat.log.gram.row.beta_hist.hist.raw.linreg.intercept,feat.log.gram.row.beta_hist.hist.raw.linreg.r2,feat.log.gram.col.beta_hist.hist.raw.mean,feat.log.gram.col.beta_hist.hist.raw.median,feat.log.gram.col.beta_hist.hist.raw.variance,feat.log.gram.col.beta_hist.hist.raw.std,feat.log.gram.col.beta_hist.hist.raw.skewness,feat.log.gram.col.beta_hist.hist.raw.kurtosis,feat.log.gram.col.beta_hist.hist.raw.entropy,feat.log.gram.col.beta_hist.hist.raw.L1_norm,feat.log.gram.col.beta_hist.hist.raw.L2_norm,feat.log.gram.col.beta_hist.hist.raw.rms,feat.log.gram.col.beta_hist.hist.raw.energy,feat.log.gram.col.beta_hist.hist.raw.zero_crossing_rate,feat.log.gram.col.beta_hist.hist.raw.autocorr_lag1,feat.log.gram.col.beta_hist.hist.raw.psd_total_power,feat.log.gram.col.beta_hist.hist.raw.linreg.slope,feat.log.gram.col.beta_hist.hist.raw.linreg.intercept,feat.log.gram.col.beta_hist.hist.raw.linreg.r2,feat.log.gram.skew.beta_hist.hist.raw.mean,feat.log.gram.skew.beta_hist.hist.raw.median,feat.log.gram.skew.beta_hist.hist.raw.variance,feat.log.gram.skew.beta_hist.hist.raw.std,feat.log.gram.skew.beta_hist.hist.raw.skewness,feat.log.gram.skew.beta_hist.hist.raw.kurtosis,feat.log.gram.skew.beta_hist.hist.raw.entropy,feat.log.gram.skew.beta_hist.hist.raw.L1_norm,feat.log.gram.skew.beta_hist.hist.raw.L2_norm,feat.log.gram.skew.beta_hist.hist.raw.rms,feat.log.gram.skew.beta_hist.hist.raw.energy,feat.log.gram.skew.beta_hist.hist.raw.zero_crossing_rate,feat.log.gram.skew.beta_hist.hist.raw.autocorr_lag1,feat.log.gram.skew.beta_hist.hist.raw.psd_total_power,feat.log.gram.skew.beta_hist.hist.raw.linreg.slope,feat.log.gram.skew.beta_hist.hist.raw.linreg.intercept,feat.log.gram.skew.beta_hist.hist.raw.linreg.r2
str,str,f64,str,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""19456""",19456.0,"""19456""",19456.0,"""19456""","""19456""",19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,…,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0,19456.0
"""null_count""","""0""",0.0,"""0""",0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",null,3.342105,null,6.552632,null,null,115.001234,0.108256,0.066683,0.025417,0.149904,4.41741,27.923921,1.188746,0.108256,0.019154,0.188448,4.77018,0.0,0.408275,1.574589,-0.001731,0.181249,0.122046,0.083922,0.0385,0.027845,0.155197,4.911607,32.

In [48]:
# print unique values for columns which start with "activations"
print(f"{DATA_RAW['activation.model'].unique() = }")

DATA_RAW['activation.model'].unique() = shape: (2,)
Series: 'activation.model' [str]
[
	"tiny-stories-1M"
	"pythia-14m"
]


In [49]:
for col in DATA_RAW.columns:
	if col.startswith("activation"):
		print(f"{col = }")

col = 'activation.model'
col = 'activation.layer'
col = 'activation.cache_key'
col = 'activation.head'
col = 'activation.cls'
col = 'activation.prompt'
col = 'activation.n_ctx'


In [56]:
nan_stats_df = nan_stats(DATA_RAW)
print(set(nan_stats_df["count"]))
nan_stats_df

{0}


feature,count
str,u32
"""activation.model""",0
"""activation.layer""",0
"""activation.cache_key""",0
"""activation.head""",0
"""activation.cls""",0
…,…
"""feat.log.gram.skew.beta_hist.h…",0
"""feat.log.gram.skew.beta_hist.h…",0
"""feat.log.gram.skew.beta_hist.h…",0


# filter nans and certain models

In [57]:
# this will filter all-nan rows
DATA_FILTERED: pl.DataFrame = filter_data(
	DATA_RAW,
	# tinystories and small pythia models look very different in embedding space, so we get rid of them
	# remove_models=["tiny-stories-1M", "pythia-14m"],
)
nan_stats(DATA_FILTERED)

Not removing any models
Removing 14/176 cols
feat.diag.zero_crossing_rate                                 μ=0.00 σ=0.00 x̃=0.00 R=[0.00,0.00] shape=19456 dtype=float64
feat.first_tok.zero_crossing_rate                            μ=0.00 σ=0.00 x̃=0.00 R=[0.00,0.00] shape=19456 dtype=float64
feat.markov_transition.time.zero_crossing_rate               μ=0.00 σ=0.00 x̃=0.00 R=[0.00,0.00] shape=19456 dtype=float64
feat.markov_transition.env.lower.slope                       μ=0.00 σ=0.00 x̃=0.00 R=[-0.00,-0.00] shape=19456 dtype=float64
feat.markov_transition.env.lower.intercept                   μ=0.00 σ=0.00 x̃=0.00 R=[0.00,0.00] shape=19456 dtype=float64
feat.markov_transition.diff.median                           μ=0.00 σ=0.00 x̃=0.00 R=[0.00,0.00] shape=19456 dtype=float64
feat.gram.row.beta_hist.hist.raw.zero_crossing_rate          μ=0.00 σ=0.00 x̃=0.00 R=[0.00,0.00] shape=19456 dtype=float64
feat.gram.col.beta_hist.hist.raw.zero_crossing_rate          μ=0.00 σ=0.00 x̃=0.00 R=[0.00,0

[ /home/miv/projects/attn/attention-motifs/attention_motifs/features/analysis.py:49 ] remove_cols = ['feat.diag.zero_crossing_rate', 'feat.first_tok.zero_crossing_rate', 'feat.markov_transition.time.zero_crossing_rate', 'feat.markov_transition.env.lower.slope', 'feat.markov_transition.env.lower.intercept', 'feat.markov_transition.diff.median', 'feat.gram.row.beta_hist.hist.raw.zero_crossing_rate', 'feat.gram.col.beta_hist.hist.raw.zero_crossing_rate', 'feat.gram.skew.beta_hist.hist.raw.zero_crossing_rate', 'feat.log.gram.row.beta_hist.hist.raw.median', 'feat.log.gram.row.beta_hist.hist.raw.zero_crossing_rate', 'feat.log.gram.col.beta_hist.hist.raw.median', 'feat.log.gram.col.beta_hist.hist.raw.zero_crossing_rate', 'feat.log.gram.skew.beta_hist.hist.raw.zero_crossing_rate']


feature,count
str,u32
"""activation.model""",0
"""activation.layer""",0
"""activation.cache_key""",0
"""activation.head""",0
"""activation.cls""",0
…,…
"""feat.log.gram.skew.beta_hist.h…",0
"""feat.log.gram.skew.beta_hist.h…",0
"""feat.log.gram.skew.beta_hist.h…",0


# scaling

In [58]:
# Pick feature columns
FEATURE_COLS: list[str] = [
	col for col in DATA_FILTERED.columns if col.startswith("feat.")
]

In [59]:
DATA_SCALED: pl.DataFrame = normalize_data(DATA_FILTERED, FEATURE_COLS)
DATA_SCALED

activation.model,activation.layer,activation.cache_key,activation.head,activation.cls,activation.prompt,activation.n_ctx,feat.diag.mean,feat.diag.median,feat.diag.variance,feat.diag.std,feat.diag.skewness,feat.diag.kurtosis,feat.diag.entropy,feat.diag.L1_norm,feat.diag.L2_norm,feat.diag.rms,feat.diag.energy,feat.diag.autocorr_lag1,feat.diag.psd_total_power,feat.diag.linreg.slope,feat.diag.linreg.intercept,feat.diag.linreg.r2,feat.first_tok.mean,feat.first_tok.median,feat.first_tok.variance,feat.first_tok.std,feat.first_tok.skewness,feat.first_tok.kurtosis,feat.first_tok.entropy,feat.first_tok.L1_norm,feat.first_tok.L2_norm,feat.first_tok.rms,feat.first_tok.energy,feat.first_tok.autocorr_lag1,feat.first_tok.psd_total_power,feat.first_tok.linreg.slope,…,feat.log.gram.row.beta_hist.hist.raw.energy,feat.log.gram.row.beta_hist.hist.raw.autocorr_lag1,feat.log.gram.row.beta_hist.hist.raw.psd_total_power,feat.log.gram.row.beta_hist.hist.raw.linreg.slope,feat.log.gram.row.beta_hist.hist.raw.linreg.intercept,feat.log.gram.row.beta_hist.hist.raw.linreg.r2,feat.log.gram.col.beta_hist.hist.raw.mean,feat.log.gram.col.beta_hist.hist.raw.variance,feat.log.gram.col.beta_hist.hist.raw.std,feat.log.gram.col.beta_hist.hist.raw.skewness,feat.log.gram.col.beta_hist.hist.raw.kurtosis,feat.log.gram.col.beta_hist.hist.raw.entropy,feat.log.gram.col.beta_hist.hist.raw.L1_norm,feat.log.gram.col.beta_hist.hist.raw.L2_norm,feat.log.gram.col.beta_hist.hist.raw.rms,feat.log.gram.col.beta_hist.hist.raw.energy,feat.log.gram.col.beta_hist.hist.raw.autocorr_lag1,feat.log.gram.col.beta_hist.hist.raw.psd_total_power,feat.log.gram.col.beta_hist.hist.raw.linreg.slope,feat.log.gram.col.beta_hist.hist.raw.linreg.intercept,feat.log.gram.col.beta_hist.hist.raw.linreg.r2,feat.log.gram.skew.beta_hist.hist.raw.mean,feat.log.gram.skew.beta_hist.hist.raw.median,feat.log.gram.skew.beta_hist.hist.raw.variance,feat.log.gram.skew.beta_hist.hist.raw.std,feat.log.gram.skew.beta_hist.hist.raw.skewness,feat.log.gram.skew.beta_hist.hist.raw.kurtosis,feat.log.gram.skew.beta_hist.hist.raw.entropy,feat.log.gram.skew.beta_hist.hist.raw.L1_norm,feat.log.gram.skew.beta_hist.hist.raw.L2_norm,feat.log.gram.skew.beta_hist.hist.raw.rms,feat.log.gram.skew.beta_hist.hist.raw.energy,feat.log.gram.skew.beta_hist.hist.raw.autocorr_lag1,feat.log.gram.skew.beta_hist.hist.raw.psd_total_power,feat.log.gram.skew.beta_hist.hist.raw.linreg.slope,feat.log.gram.skew.beta_hist.hist.raw.linreg.intercept,feat.log.gram.skew.beta_hist.hist.raw.linreg.r2
str,i64,str,i64,str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""pythia-14m""",0,"""blocks.0.attn.hook_pattern""",0,"""pythia-14m:L0:H0""","""MjWkadUxH4Z5nfLEFtUc1w""",114,0.694321,0.623202,0.076438,0.278822,-1.197101,-0.975054,1.385185,0.694321,0.254704,0.498966,0.220855,-0.77136,0.006785,0.060889,0.902772,-0.198222,-0.069683,-0.251118,0.032948,0.234346,-0.669495,-0.710484,0.490211,-0.069683,-0.104345,0.024284,-0.078127,-0.307481,-0.014812,0.091667,…,0.064931,0.567836,-0.453251,-0.397889,0.397286,-0.341781,-0.499745,0.125569,0.180488,0.681093,0.722313,0.157006,-0.499745,0.168357,0.168357,0.117291,0.520078,-0.278334,-0.49915,0.394283,-0.424271,-0.27325,0.100127,-0.329232,-0.306534,0.237982,0.241786,-0.403167,-0.27325,-0.311198,-0.311198,-0.33081,-0.240143,-0.343456,0.359227,-0.344892,-0.391807
"""pythia-14m""",0,"""blocks.0.attn.hook_pattern""",1,"""pythia-14m:L0:H1""","""MjWkadUxH4Z5nfLEFtUc1w""",114,1.609187,0.57747,2.307061,2.461048,-1.655127,-1.239506,2.00635,1.609187,1.385089,1.971433,1.58831,-1.558366,1.85054,0.618968,1.124181,-1.315402,-0.403627,-0.309852,-0.152398,-0.023133,0.126941,-0.174762,-0.485918,-0.403627,-0.279698,-0.209546,-0.191723,-3.710731,-0.238823,0.434141,…,1.683613,0.678884,-1.486883,-1.531702,1.532559,-1.62

In [61]:
nan_stats(DATA_SCALED)

feature,count
str,u32
"""activation.model""",0
"""activation.layer""",0
"""activation.cache_key""",0
"""activation.head""",0
"""activation.cls""",0
…,…
"""feat.log.gram.skew.beta_hist.h…",0
"""feat.log.gram.skew.beta_hist.h…",0
"""feat.log.gram.skew.beta_hist.h…",0


# PCA

In [ ]:
# PCA
pca_data: np.ndarray
pca_obj: PCA
pca_data, pca_obj = apply_pca(DATA_SCALED, n_components=10, feature_cols=FEATURE_COLS)

ValueError: Input X contains NaN.
PCA does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
# save some data
DATA_SCALED.write_ndjson(Path("../data/features/scaled.jsonl"))
dbg_tensor(pca_data)
np.save("../data/features/pca.npy", pca_data)

# only columns that start with "activation."
# for labelling the PCA plot in the webapp
data_meta: pl.DataFrame = DATA_SCALED[[
	col for col in DATA_SCALED.columns if col.startswith("activation.")
]]
data_meta = data_meta.write_ndjson(Path("../data/features/meta.jsonl"))

In [ ]:
for i in range(1, 3):
	for j in range(0, i):
		plot_embedding(
			pca_data,
			DATA_SCALED["activation.model"],
			(i, j),
			alpha=0.05,
			marker_size=10,
			title=f"2D PCA Embedding ({i}, {j})",
		)